**LangChain Agents Intro**

angChain is one of the most popular open source libraries for AI Engineers. It's goal is to abstract away the complexity in building AI software, provide easy-to-use building blocks, and make it easier when switching between AI service providers.

In this example, we will introduce LangChain's Agents, adding the ability to use tools such as search and calculators to complete tasks that normal LLMs cannot fufil. In this example we will be using OpenAI's gpt-4o-mini.

In [3]:
!pip install google-search-results 
#allow the llm to use web search results to answer questions about current events or to find information that is not in its training data.


[notice] A new release of pip available: 22.2.2 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import langchain_community
import langchain_core
import langchain
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.1",
    temperature=0
)

c:\Users\PatelDharmikkumar\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


**Introduction to Tools**

Tools are a way augment our LLMs with code execution. A tool is simply a function formatted so that our agent can undertstand how to use it, and then execute it. Let's start by creating a few simple tools.

We can use the @tool decorator to create an LLM-compatible tool from a standard python function — this function should include a few things for optimal performance:

+ A docstring describing what the tool does and when it should be used, this will be read by our LLM/agent and used to decide when to use the tool, and also how to use the tool.
+ Clear parameter names that ideally tell the LLM what each parameter is, if it isn't clear we make sure the docstring explains what the parameter is for and how to use it.
+ Both parameter and return type annotations.

In [17]:
from langchain_core.tools import tool

@tool # the decorator makes this a structured tool object that the agent can use
def add(x: float, y: float) -> float:
    """Add 'x' and 'y'."""
    return x + y

@tool
def multiply(x: float, y: float) -> float:
    """Multiply 'x' and 'y'."""
    return x * y

@tool
def exponentiate(x: float, y: float) -> float:
    """Raise 'x' to the power of 'y'."""
    return x ** y

@tool
def subtract(x: float, y: float) -> float:
    """Subtract 'x' from 'y'."""
    return y - x

With the @tool decorator our function is turned into a StructuredTool object, which we can see below:

In [18]:
add

StructuredTool(name='add', description="Add 'x' and 'y'.", args_schema=<class 'langchain_core.utils.pydantic.add'>, func=<function add at 0x0000017B58705B40>)

We can see the tool name, description, and arg schema:

In [19]:
print(f"{add.name=}\n{add.description=}")

add.name='add'
add.description="Add 'x' and 'y'."


we can see the json schema of the object

In [20]:
add.args_schema.model_json_schema()

{'description': "Add 'x' and 'y'.",
 'properties': {'x': {'title': 'X', 'type': 'number'},
  'y': {'title': 'Y', 'type': 'number'}},
 'required': ['x', 'y'],
 'title': 'add',
 'type': 'object'}

In [21]:
exponentiate.args_schema.model_json_schema()

{'description': "Raise 'x' to the power of 'y'.",
 'properties': {'x': {'title': 'X', 'type': 'number'},
  'y': {'title': 'Y', 'type': 'number'}},
 'required': ['x', 'y'],
 'title': 'exponentiate',
 'type': 'object'}

When invoking the tool, a JSON string output by the LLM will be parsed into JSON and then consumed as kwargs, similar to the below:

In [22]:
import json

llm_output_string = "{\"x\": 5, \"y\": 2}"  # this is the output from the LLM
llm_output_dict = json.loads(llm_output_string)  # load as dictionary
llm_output_dict

{'x': 5, 'y': 2}

This is then passed into the tool function as kwargs (keyword arguments) as indicated by the ** operator - the ** operator is used to unpack the dictionary into keyword arguments.

In [23]:
exponentiate.func(**llm_output_dict)

25

**Creating an Agent**

We're going to construct a simple tool calling agent. We will use LangChain Epression Language (LCEL) to construct the agent. We will cover LCEL more in the next chapter, but for now - all we need to know is that our agent will be constructed using syntax and components like so:

        agent = (
            <input parameters, including chat history and user query>
            | <prompt>
            | <LLM with tools>
        )

We need this agent to remember previous interactions within the conversation. To do that, we will use the ChatPromptTemplate with a system message, a placeholder for our chat history, a placeholder for the user query, and finally a placeholder for the agent scratchpad.

The agent scratchpad is where the agent will write it's "notes" as it is working through multiple internal thought and tool-use steps to produce a final output to the user.

In [24]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "you're a helpful assistant"),
    MessagesPlaceholder(variable_name="chat_history"), #placeholder for the chat history memory
    ("human", "{input}"),
    ("placeholder", "{agent_scratchpad}"),
])

When creating an agent we need to add conversational memory to make the agent remember previous interactions. We'll be using the older ConversationBufferMemory class rather than the newer RunnableWithMessageHistory — the reason being that we will also be using the older create_tool_calling_agent and AgentExecutor method and class.

In [26]:
from langchain_community.chat_message_histories import ChatMessageHistory

memory = ChatMessageHistory(memory_key="chat_history", return_messages=True) # the memory_key is the variable name used in the prompt, return_messages=True means that the memory will return a list of messages instead of a string summary
memory.add_user_message("Hi, I am learning LangChain!")
memory.add_ai_message("That's awesome!")

print(memory.messages)

[HumanMessage(content='Hi, I am learning LangChain!', additional_kwargs={}, response_metadata={}), AIMessage(content="That's awesome!", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


Now we will initialize our agent. For that we need:

+ llm: as already defined
+ tools: to be defined (just a list of our previously defined tools)
+ prompt: as already defined
+ memory: as already defined

In [32]:
from langchain.agents import create_agent

tools = [add, subtract, multiply, exponentiate]

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="You are a helpful assistant",
)

In [40]:
output = agent.invoke({
    "messages": [
        {"role": "user", "content": "Use the multiply tool to calculate 10.7 * 7.68"}
    ]
})

for msg in output["messages"]:
    print(type(msg).__name__, getattr(msg, "tool_calls", None), msg.content)

HumanMessage None Use the multiply tool to calculate 10.7 * 7.68
AIMessage [{'name': 'multiply', 'args': {'x': 10.7, 'y': 7.68}, 'id': '4e7deb09-12dc-4b3d-89ef-f988db67f632', 'type': 'tool_call'}] 
ToolMessage None 82.17599999999999
AIMessage [] The result of multiplying 10.7 by 7.68 is 82.176.


Here, we can see the LLM has generated that we should use the multiply tool and the tool input should be {"x": 10.7, "y": 7.68}. However, the tool is not executed. For that to happen we need an agent execution loop, which will handle the multiple iterations of generation to tool calling to generation, etc.


Let's test our agent with some memory and tool use. First, we tell it our name, then we will perform a few tool calls, then see if the agent can still recall our name.

In [50]:
output = agent.invoke({
    "messages": memory.messages + [
        {"role": "user", "content": "Hi! My name is James"}
    ]
})

memory.add_user_message("Hi! My name is James")
memory.add_ai_message(output["messages"][-1].content)

print(output["messages"][-1].content)

Nice to meet you again, James! It looks like we've just added 0 + 0 = 0. What's next?


Now let's try and get the agent to perform multiple tool calls within a single execution loop:

In [51]:
output = agent.invoke({
    "messages": memory.messages + [
        {"role": "user", "content": "What is nine plus 10, minus 4 * 2, to the power of 3"}
    ]
})

memory.add_user_message("What is nine plus 10, minus 4 * 2, to the power of 3")
memory.add_ai_message(output["messages"][-1].content)

print(output["messages"][-1].content)

So, the final result is 19 + 9 - 4 * 2 to the power of 3 = 19 + 9 - 8^3 = 19 + 9 - 512 = -484.


In [52]:
output = agent.invoke({
    "messages": memory.messages + [
        {"role": "user", "content": "what is my name?"}
    ]
})

memory.add_user_message("what is my name?")
memory.add_ai_message(output["messages"][-1].content)

print(output["messages"][-1].content)

Your name is James.


The agent has successfully recalled our name. Let's move on to another agent example.